# 動態 RORO 風險管理與跨資產配置策略 - 主工作流程

本 Notebook 實現動態風險開啟/關閉 (Risk-On/Risk-Off, RORO) 機制，結合市場寬度指標和系統壓力指數，
動態調整台灣加權股價指數期貨 (TXF)、美國長天期公債 ETF (TLT) 和黃金 ETF (GLD) 的配置。

## 專案目標
- 將整體投資組合的最大回撤 (MDD) 嚴格控制在 10% 以內
- 追求長期、可持續的正報酬
- 實現相對於台灣加權股價指數的低 Beta (<1)
- 在風險調整後爭取產生超額回報

## 核心方法論
RORO 信號生成基於：
- 市場寬度指標 (台灣股市騰落線相關)
- 一級交易商壓力指數 (SOFR、利差、持倉部位、MOVE、VIX、準備金比率)

資產配置邏輯：
- Risk-On：增加 TXF 多頭部位，減少避險資產
- Risk-Off：減少 TXF，增加 TLT 和 GLD
- 中性：維持較低風險曝險

## 開發階段
階段一：環境與數據管道搭建
階段二：指標與信號開發
階段三：初步策略回測
階段四：迭代優化
階段五：穩健性檢驗

In [ ]:
# Cell 1: 環境設置與模組導入
import sys
import os
from pathlib import Path

# 設置專案根目錄到 Python 路徑
project_root = Path.cwd().parent  # 假設 notebook 在 notebooks/ 子目錄中
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# 匯入自定義模組
try:
    from src.config_manager import ConfigManager
    from src.data_loader import DataLoader
    print("✅ 自定義模組匯入成功")
except ImportError as e:
    print(f"❌ 自定義模組匯入失敗: {e}")
    print("請檢查 src/ 目錄中的模組是否正確實現")

# 匯入第三方函式庫
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 設置圖表樣式
plt.style.use('default')
sns.set_palette('husl')

print("✅ 環境設置完成")

In [ ]:
# Cell 2: 配置管理設置
from pathlib import Path

# 設定文件路徑
config_dir = project_root / 'config'
data_dir = project_root / 'data'

# 建立必要的目錄
config_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)

# 初始化配置管理器
# 注意：在實際使用時，需要先建立 config/strategy_config.yaml 文件
config_path = config_dir / 'strategy_config.yaml'
if config_path.exists():
    config_manager = ConfigManager(config_path)
    print("✅ 配置管理器初始化成功")
else:
    print("⚠️ 配置檔案不存在，將使用默認設定")
    config_manager = None

# 設定默認參數
DEFAULT_PARAMS = {
    'data': {
        'start_date': '2020-01-01',
        'end_date': None,  # None 表示到最新日期
        'tickers': {
            'txf': 'TXF.TW',  # 台灣期貨指數期貨
            'tlt': 'TLT',     # 美國長天期公債 ETF
            'gld': 'GLD'      # 黃金 ETF
        }
    },
    'strategy': {
        'max_drawdown_limit': 0.10,  # 10% MDD 限制
        'beta_target': 0.8,          # 目標 Beta
        'rebalance_frequency': 'daily'  # 再平衡頻率
    },
    'backtest': {
        'initial_capital': 1000000,  # 初始資本 100 萬
        'transaction_cost': 0.001    # 交易成本 0.1%
    }
}

print("✅ 配置設置完成")

In [ ]:
# Cell 3: 數據載入器初始化與測試
from src.data_loader import DataLoader

# 初始化數據載入器
data_loader = DataLoader(data_dir)
print("✅ 數據載入器初始化成功")

# 測試 Yahoo Finance 數據載入功能
print("\n🔄 測試數據載入功能...")

# 測試載入 TXF 數據（台灣期貨指數）
try:
    txf_data = data_loader.load_yahoo_finance_data(
        ticker='^TWII',  # 台灣加權指數
        start_date='2023-01-01'
    )
    if not txf_data.empty:
        print(f"✅ TXF 數據載入成功，數據形狀: {txf_data.shape}")
        print(f"數據日期範圍: {txf_data.index.min()} 到 {txf_data.index.max()}")
    else:
        print("❌ TXF 數據載入失敗或返回空數據")
except Exception as e:
    print(f"❌ TXF 數據載入測試失敗: {e}")

# 測試載入 TLT 數據
try:
    tlt_data = data_loader.load_yahoo_finance_data(
        ticker='TLT',
        start_date='2023-01-01'
    )
    if not tlt_data.empty:
        print(f"✅ TLT 數據載入成功，數據形狀: {tlt_data.shape}")
    else:
        print("❌ TLT 數據載入失敗或返回空數據")
except Exception as e:
    print(f"❌ TLT 數據載入測試失敗: {e}")

# 測試載入 GLD 數據
try:
    gld_data = data_loader.load_yahoo_finance_data(
        ticker='GLD',
        start_date='2023-01-01'
    )
    if not gld_data.empty:
        print(f"✅ GLD 數據載入成功，數據形狀: {gld_data.shape}")
    else:
        print("❌ GLD 數據載入失敗或返回空數據")
except Exception as e:
    print(f"❌ GLD 數據載入測試失敗: {e}")

print("\n🎯 數據載入測試完成")

## 下一步開發計劃

### 階段一：環境與數據管道（當前階段）
- ✅ 專案目錄結構搭建
- ✅ 基礎模組開發 (ConfigManager, DataLoader)
- 🔄 數據獲取與快取功能實現
- 🔄 數據處理與 Parquet 儲存

### 階段二：指標與信號開發（下一階段）
- 市場寬度指標計算
- 一級交易商壓力指數整合
- RORO 狀態判斷邏輯

### 階段三：策略回測框架
- 資產配置邏輯實現
- 風險控制機制
- 績效評估指標

### 階段四：優化與驗證
- 參數敏感性分析
- 樣本外測試
- 蒙地卡羅模擬

### 階段五：實盤考量
- 執行層面優化
- 監控儀表板開發
- 風險管理強化

In [ ]:
# Cell 4: 開發日誌記錄
from datetime import datetime

# 記錄開發進度
development_log = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'stage': '階段一：環境與數據管道',
    'completed_tasks': [
        '專案目錄結構搭建',
        'ConfigManager 類別實現',
        'DataLoader 類別基礎框架',
        '主 Notebook 環境設置'
    ],
    'next_tasks': [
        '實現 DataLoader 中獲取 TXF, TLT, GLD 價格數據並整合 requests-cache',
        '實現將這些價格數據處理後儲存為 Parquet 檔案的邏輯',
        '開發市場寬度指標計算模組',
        '整合一級交易商壓力指數'
    ],
    'notes': '已完成基礎環境搭建，數據載入功能測試中。準備進入指標開發階段。'
}

# 顯示開發日誌
print("📋 開發日誌：")
for key, value in development_log.items():
    if key != 'notes':
        print(f"{key}: {value}")
    else:
        print(f"\n{key}:\n{value}")

# 保存開發日誌到文件
log_file = project_root / 'docs' / 'development_log.txt'
log_file.parent.mkdir(exist_ok=True)

with open(log_file, 'a', encoding='utf-8') as f:
    f.write(f"\n--- {development_log['timestamp']} ---\n")
    for key, value in development_log.items():
        f.write(f"{key}: {value}\n")

print(f"\n✅ 開發日誌已保存到 {log_file}")